# Notebook 0: Data Quality Check and Cleaning

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

Wczytanie danych

In [3]:
df = pd.read_csv('data/zbiór_2.csv')

In [4]:
df.head()

,szczegolnaFormaPrawna_Symbol,formaWlasnosci_Symbol,pkdKod,wsk_liczba_dni_istnienia,Aktywa,Aktywa_trwale,Wartosci_niematerialne_prawne,Wartosc_firmy,Rzeczowe_aktywa_trwale,Srodki_trwale,...,wsk_struktura_kap_wlasnego_s_1,wsk_struktura_kap_wlasnego_s_2,wsk_zadluzenia,wsk_zob_dlugoterminowe_aktywa_rzeczowe,wsk_zob_oprocentowanych,wsk_zob_oprocentowanych_aktywa_rzeczowe,wsk_struktura_kap_obcego_s,wsk_zob_s_aktywa_rzeczowe,wsk_fin_majatku_kapitalem,default
0,117,224,4120,1501,667091.16,87750.43,0.0,0.0,87750.43,87750.43,...,0.063976,0.080032,14.630797,0.000000,0.000000,0.000000,0.936024,0.735403,-0.077800,1
1,117,214,6820,1570,180157.00,0.00,0.0,0.0,0.00,0.00,...,0.993339,0.993339,0.006706,0.006661,0.000000,0.000000,0.006661,0.006661,0.993339,1
2,117,0,4646,1952,297880.73,43721.66,0.0,0.0,43721.66,43721.66,...,0.199896,0.245042,4.002593,0.500066,0.500066,0.500066,0.800104,0.615868,0.062259,1
3,117,214,7010,1361,11599.10,100.00,0.0,0.0,0.00,0.00,...,0.655146,4.095093,0.526378,0.000000,0.344854,0.344854,0.344854,-0.495163,0.652147,1
4,117,216,6201,2170,45883.96,1847.82,0.0,0.0,0.00,0.00,...,0.309014,0.309014,2.236101,0.690986,0.000000,0.000000,0.690986,0.690986,0.280019,1


## Feature overview
- Krótkie spojrzenie na wybrane cechy (może być pomocne przy przeglądzie jakości danych, analizie wykresów oraz do dalszej interpretacji).
---
#### 1. Rozmiar i Struktura Majątku (Bilans - Aktywa):
- Te zmienne mówią o tym, jak duża jest firma i co posiada oraz o potencjale produkcyjnym firmy i jej elastyczności.
- <strong>Aktywa</strong> (Suma bilansowa): Całkowita wartość majątku firmy.
    - Intuicja: Im większe aktywa, tym zazwyczaj stabilniejsza firma (efekt skali), ale trudniej nią sterować.
- <strong>Aktywa_trwale</strong> (Rzeczowe aktywa trwałe): Budynki, maszyny, grunty, licencje. To, czego nie da się szybko sprzedać.
    - Intuicja: Wysoki udział oznacza stabilność (duży majątek produkcyjny), ale niską płynność i małą elastyczność w kryzysie (trudno sprzedać fabrykę w tydzień, by spłacić długi).
- <strong>Aktywa_obrotowe</strong> (Zapasy, należności, gotówka): "Paliwo" bieżącej działalności.
    - Intuicja: Muszą rotować i być na tyle duże, by firma mogła działać płynnie, ale nie za duże (żeby nie mrozić gotówki).
- <strong>Zapasy</strong>: Surowce, towar w magazynie.
    - Ryzyko: Jeśli rosną szybciej niż sprzedaż -> firma produkuje "na magazyn", brak zbytu) = problem z gotówką.
- <strong>Naleznosci_krotkoterminowe</strong>: Faktury wystawione klientom, jeszcze nieopłacone.
    - Ryzyko: Jeśli rosną drastycznie -> klienci przestali płacić. To tzw. "kredyt kupiecki" udzielany przez firmę innym.
- <strong>Inwestycje_krotkoterminowe</strong>: Zazwyczaj gotówka w kasie i na rachunkach.
    - Intuicja: Poduszka bezpieczeństwa. Im więcej, tym lepiej dla spłaty kredytu (ale jeśli za dużo, to firma nie inwestuje w rozwój).
- <strong>Inwestycje_dlugoterminowe</strong>: Udziały w innych spółkach, nieruchomości inwestycyjne.
    - Intuicja: Skarbonka na przyszłość, ale często trudna do wyceny i upłynnienia.
---
#### 2. Struktura Finansowania (Bilans - Pasywa)
- Te zmienne mówią, za czyje pieniądze firma kupiła majątek (swoje czy pożyczone).
- <strong>Kapital_wlasny</strong>: Wkład właścicieli + skumulowane zyski z lat ubiegłych.
    - Ekonomia: To jest bufor bezpieczeństwa. Jeśli firma ma straty, pokrywa je z tego kapitału. Im wyższy, tym bezpieczniejszy kredyt. Jeśli ujemny -> firma jest bankrutem (pasywa > aktywa).
- <strong>Kapital_podstawowy</strong>: Pierwotny wkład założycieli.
    - Intuicja: Zazwyczaj stały. Jego nagłe zmiany (podwyższenie) mogą oznaczać ratowanie spółki przez właścicieli.
- <strong>Kapital_zapasowy</strong>: Część zysku odłożona "na czarną godzinę".
    - Intuicja: Im wyższy, tym firma bardziej odporna na szoki rynkowe.
- <strong>Zobowiazania_krotkoterminowe</strong> (do 12 m-cy): Faktury do zapłacenia, podatki, raty kredytów na ten rok.
    - Ryzyko: Najbardziej niebezpieczna część pasywów. Jeśli przewyższają aktywa obrotowe, firma traci płynność.
- <strong>Zobowiazania_dlugoterminowe</strong> (pow. 1 roku): Kredyty inwestycyjne, obligacje.
    - Intuicja: "Zdrowy" dług służący rozwojowi. Mniej groźny niż krótki, bo bank nie zażąda spłaty jutro.
- <strong>Kredyty_pozyczki</strong>: Część zobowiązań, która jest oprocentowana (dług bankowy).
    - Ryzyko: Bezpośrednie obciążenie kosztami finansowymi.
- <strong>Rezerwy_na_zobowiazania</strong>: Odłożone środki na przyszłe, pewne koszty (np. sprawy sądowe, odprawy).
    - Intuicja: Konserwatyzm księgowy. Duże rezerwy mogą zwiastować nadchodzące problemy prawne.
---
#### 3. Wyniki i Efektywność (Rachunek Zysków i Strat - RZiS)
- Te zmienne mówią, czy firma zarabia na swojej działalności.
- <strong>Przychody_netto_ze_sprzedazy</strong>: Obrót firmy.
    - Intuicja: Dynamika przychodów to puls firmy. Spadek r/r to pierwszy sygnał utraty rynku.
- <strong>Zysk_ze_sprzedazy</strong>: Przychody minus bezpośrednie koszty wytworzenia.
    - Intuicja: Marża podstawowa. Jeśli ujemna -> firma dokłada do każdego sprzedanego produktu.
- <strong>Koszty_operacyjne</strong>: Ile kosztuje wytworzenie produktu/usługi (wynagrodzenia, zużycie materiałów, usługi obce).
    - Intuicja: Baza kosztowa. Jej sztywność decyduje o wrażliwości na kryzys.
- <strong>Amortyzacja</strong>: Koszt niepieniężny (zużycie majątku, np. maszyn).
    - Intuicja: Dodajemy ją do zysku, by oszacować EBITDA (gotówkę operacyjną). Wysoka amortyzacja = duże inwestycje w przeszłości.
- <strong>Koszty_finansowe / koszty_odsetki</strong>: Koszt obsługi długu (ile firma płaci bankom).
    - Ryzyko: Jeśli rosną przy stałym zadłużeniu -> wzrosły stopy procentowe lub bank podniósł marżę (uznał firmę za ryzykowną).
- <strong>Zysk_netto</strong>: Wynik końcowy ("Bottom line").
    - Intuicja: To co zostaje dla właścicieli. Podstawa do budowania kapitału własnego.
- <strong>Dotacje</strong>: Pozostałe przychody operacyjne z dotacji (UE, państwo).
    - Ryzyko: Zysk "papierowy" lub jednorazowy. W modelu ratingowym traktuje się to ostrożnie - firma uzależniona od dotacji jest niestabilna ratingowo.
---
#### 4. Wskaźniki Finansowe (Kluczowe w modelu)
- Gotowe miary, które zazwyczaj mają największą moc predykcyjną (Information Value).
- <strong>wsk_plynnosci_biezacej</strong> (Current Ratio): Aktywa Obrotowe / Zobowiązania Krótkie.
    - Intuicja: Ile razy majątek obrotowy pokrywa pilne długi. Norma: 1.2 - 2.0. Poniżej 1.0 -> zagrożenie upadłością.
- <strong>wsk_plynnosci_szybkiej</strong> (Quick Ratio): (Aktywa Obrotowe - Zapasy) / Zobowiązania Krótkie.
    - Intuicja: Czy firma spłaci długi, jeśli nie uda jej się sprzedać towaru z magazynu? Bardziej rygorystyczny test.
- <strong>wsk_zadluzenia_ogolnego</strong>: Zobowiązania Ogółem / Aktywa.
    - Ryzyko: Stopień lewarowania. Powyżej 0.7-0.8 firma należy w większości do wierzycieli, a nie właścicieli.
- <strong>wsk_rentownosci_sprzedazy_netto</strong> (ROS): Zysk Netto / Przychody.
    - Intuicja: Ile groszy zostaje w kieszeni z każdej złotówki obrotu. Niska marża = mały margines błędu.
- <strong>wsk_rentownosci_aktywow</strong> (ROA): Zysk Netto / Aktywa.
    - Intuicja: Efektywność "fabryki". Jak dobrze majątek pracuje na zysk.
- <strong>wsk_cykl_rotacji_naleznosci</strong> (DSO): Średnia liczba dni oczekiwania na zapłatę od klienta.
    - Ryzyko: Im wyższy, tym gorzej (zamrażanie gotówki). Nagły wzrost = klienci przestają płacić.
- <strong>wsk_cykl_rotacji_zapasow</strong> (DSI): Średnia liczba dni leżakowania towaru.
  - Ryzyko: Wzrost oznacza, że towar się nie sprzedaje (przestarzały) lub nadprodukcję.
- <strong>wsk_pokrycia_odsetek</strong> (Interest Coverage): Zysk Operacyjny (lub EBITDA) / Odsetki.
    - Intuicja: Ile razy zysk pokrywa ratę odsetkową. Wartość < 1.0 to "strefa śmierci" (firma musi pożyczać na spłatę odsetek).
- <strong>wsk_zysk_ebitda_...</strong>: Różne wariacje marży EBITDA.
    - Intuicja: Najlepsza miara zdolności do generowania gotówki operacyjnej, niezależna od polityki amortyzacyjnej i podatkowej.
---
#### 5. Inne / Metadane
- Zmienne jakościowe.
- <strong>formaWlasnosci_Symbol</strong>: Status prawny (np. Sp. z o.o., S.A., Sp. Jawna).
    - Ryzyko: Spółki kapitałowe (z o.o., S.A.) są zazwyczaj bezpieczniejsze i mają pełną księgowość. Działalności gospodarcze są bardziej ryzykowne.
- <strong>wsk_liczba_dni_istnienia</strong>: Lata na rynku.
    - Intuicja: "Efekt Lindy" – im dłużej firma istnieje, tym większa szansa, że przetrwa kolejny rok. Start-upy (<2

In [5]:
def data_quality_report(df: pd.DataFrame):
    report = {}

    # --- 1️⃣ Podstawowe metryki ---
    report['basic_info'] = {
        'num_rows': len(df),
        'num_columns': len(df.columns),
        'duplicate_rows': df.duplicated().sum(),
        'missing_values_total': df.isna().sum().sum(),
        'missing_values_percent': round(df.isna().sum().sum() / (df.size) * 100, 2)
    }

    # --- 2️⃣ Braki danych ---
    report['missing_by_column'] = (
        df.isna().mean().round(3) * 100
    ).to_dict()

    # --- 3️⃣ Anomalie / obserwacje odstające ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    outlier_summary = {}
    for col in numeric_cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        outlier_summary[col] = {
            'num_outliers': int(outliers),
            'percent_outliers': round(outliers / len(df) * 100, 2)
        }
    report['outliers'] = outlier_summary

    # --- 5️⃣ Krótkie podsumowanie tekstowe ---
    summary = f"""
    🔍 DATA QUALITY REPORT
    - Liczba wierszy: {report['basic_info']['num_rows']}
    - Liczba kolumn: {report['basic_info']['num_columns']}
    - Duplikaty: {report['basic_info']['duplicate_rows']}
    - Braki danych (łącznie): {report['basic_info']['missing_values_percent']}%
    - Kolumny numeryczne z anomaliami: {[col for col, v in outlier_summary.items() if v['num_outliers'] > 0]}
    """
    print(summary)
    return report


In [6]:
report = data_quality_report(df)


    🔍 DATA QUALITY REPORT
    - Liczba wierszy: 3000
    - Liczba kolumn: 220
    - Duplikaty: 0
    - Braki danych (łącznie): 3.16%
    - Kolumny numeryczne z anomaliami: ['formaWlasnosci_Symbol', 'pkdKod', 'wsk_liczba_dni_istnienia', 'Aktywa', 'Aktywa_trwale', 'Wartosci_niematerialne_prawne', 'Wartosc_firmy', 'Rzeczowe_aktywa_trwale', 'Srodki_trwale', 'Naleznosci_dlugoterminowe', 'Inwestycje_dlugoterminowe', 'Rozliczenia_miedzyokresowe_dlugie', 'Aktywa_obrotowe', 'Zapasy', 'Naleznosci_krotkoterminowe', 'Naleznosci_dostaw_uslug_12m_powiazane', 'Naleznosci_dostaw_uslug_pow12m_powiazane', 'Naleznosci_dostaw_uslug_12m_kapitale', 'Naleznosci_dostaw_uslug_pow12m_kapitale', 'Naleznosci_dostaw_uslug_12m_pozostale', 'Naleznosci_dostaw_uslug_pow12m_pozostale', 'Naleznosci_dostaw_uslug_pozostale_sadowe', 'Inwestycje_krotkoterminowe', 'Srodki_pieniezne', 'Rozliczenia_miedzyokresowe_krotkie', 'Kapital_wlasny', 'Kapital_podstawowy', 'Kapital_zapasowy', 'Zysk_netto', 'Zobowiazania_rezerwy', 'Rezer

e:\Programy\Anaconda\envs\interpretability\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


In [7]:
def generate_schema_docs(df):
    print("| Nazwa Kolumny | Typ Danych | Min | Max | Braki (%) | Przykładowa Wartość |")
    print("|---|---|---|---|---|---|")
    
    for col in df.columns:
        dtype = str(df[col].dtype)
        
        # Obsługa liczb
        if 'float' in dtype or 'int' in dtype:
            min_val = round(df[col].min(), 2)
            max_val = round(df[col].max(), 2)
            example = round(df[col].mean(), 2)
        else:
            min_val = "-"
            max_val = "-"
            example = df[col].iloc[0]

        missing_pct = round(df[col].isna().mean() * 100, 2)
        
        print(f"| `{col}` | {dtype} | {min_val} | {max_val} | {missing_pct}% | {example} |")

# Uruchom na swoim przetworzonym DataFrame
generate_schema_docs(df)

| Nazwa Kolumny | Typ Danych | Min | Max | Braki (%) | Przykładowa Wartość |
|---|---|---|---|---|---|
| `szczegolnaFormaPrawna_Symbol` | int64 | 117 | 117 | 0.0% | 117.0 |
| `formaWlasnosci_Symbol` | int64 | 0 | 235 | 0.0% | 200.86 |
| `pkdKod` | int64 | 0 | 9609 | 0.0% | 5357.26 |
| `wsk_liczba_dni_istnienia` | int64 | 1179 | 11955 | 0.0% | 4021.04 |
| `Aktywa` | float64 | 0.0 | 540666099.5 | 0.0% | 4884415.92 |
| `Aktywa_trwale` | float64 | 0.0 | 513408881.69 | 0.0% | 3076991.22 |
| `Wartosci_niematerialne_prawne` | float64 | 0.0 | 40302000.0 | 0.0% | 55899.96 |
| `Wartosc_firmy` | float64 | 0.0 | 3621061.0 | 0.0% | 3567.97 |
| `Rzeczowe_aktywa_trwale` | float64 | 0.0 | 512636428.9 | 0.0% | 1467481.78 |
| `Srodki_trwale` | float64 | 0.0 | 512441055.24 | 0.0% | 1272440.84 |
| `Naleznosci_dlugoterminowe` | float64 | 0.0 | 92952760.3 | 0.0% | 69348.17 |
| `Inwestycje_dlugoterminowe` | float64 | 0.0 | 458183801.45 | 0.0% | 1251658.89 |
| `Rozliczenia_miedzyokresowe_dlugie` | float64 | 0

In [8]:
df.shape

(3000, 220)

Podział na zmienne numeryczne i kategoryczne

In [9]:
NUMERIC_COLS = df.select_dtypes(include=np.number).columns.to_list()
CATEGORICAL_COLS = [col for col in df.columns.to_list() if col not in NUMERIC_COLS]

Zmienne kategoryczne są zbędne, można je wyrzucić

In [10]:
CATEGORICAL_COLS

['schemat_wsk_bilans', 'schemat_wsk_rzis']

Zmienne numeryczne, ale symboliczne, więc do przekształcenia

In [11]:
SYMBOLS = ['formaWlasnosci_Symbol', "pkdKod", 'szczegolnaFormaPrawna_Symbol']

Cała kolumna to jedna wartość - kolumna niepotrzebna

In [12]:
df['szczegolnaFormaPrawna_Symbol'].value_counts()

szczegolnaFormaPrawna_Symbol
117    3000
Name: count, dtype: int64

Jak wygląda pkdKod?

In [13]:
df['pkdKod'].value_counts()

pkdKod
6820    140
7022    121
4110    106
6920    105
4120     97
       ... 
7721      1
1082      1
6430      1
8541      1
4753      1
Name: count, Length: 388, dtype: int64

In [14]:
df['pkdKod'].unique()

array([4120, 6820, 4646, 7010, 6201, 4633, 4671, 3511, 8690, 7112,  142,
        111, 4110, 7022, 2562, 3320, 1623, 7120, 4772, 4672, 8010, 6209,
       6202, 8559, 7830, 7311, 4799, 7220, 2042, 4652, 4941, 4771, 9311,
       4791, 6420, 2110, 6629, 7111, 5520, 7810, 2899, 4511, 4299, 8211,
       1812, 1105, 6622, 8610, 2014, 6810, 4639, 2511, 8121, 4632, 4774,
       9522, 6311, 9004, 5610, 4673, 6399, 3821, 5920, 4615, 6312, 3109,
       4690, 1101, 6831, 8891, 9523, 6630, 9609, 3312, 4619, 4643, 4631,
       5621, 7820, 8560, 5814, 6130, 4311, 4399,  149, 8622, 8299, 7912,
       4675, 4677, 4520,  812, 9329, 4329, 5510, 4211, 5819, 7430, 1414,
       4623, 6920, 4751, 6832, 4649, 8129,  811, 2740, 1629, 2361, 5911,
       1624, 7721, 5811, 5229,  990, 4939,  161, 7490, 8623, 8810, 4532,
       4730, 2052, 3299, 8621, 3240, 2593, 8122, 2572, 4773, 4719, 4777,
       6619, 1092, 7219, 4322, 5310, 4618, 4531, 6499, 8291, 4642, 6492,
       6910, 4711,  129, 6110, 2229, 2892, 2331, 46

Jak wygląda forma własności?

In [15]:
df['formaWlasnosci_Symbol'].value_counts()

formaWlasnosci_Symbol
214    1987
215     285
216     260
0       175
224      90
226      54
225      48
227      38
113      33
112      12
132       5
234       4
133       3
235       3
111       1
121       1
123       1
Name: count, dtype: int64

In [16]:
df['formaWlasnosci_Symbol'].unique()

array([224, 214,   0, 216, 215, 225, 112, 226, 113, 227, 234, 133, 111,
       132, 235, 121, 123], dtype=int64)

Przerobienie pkdkod i formy własności na zmienne kategoryczne, potem zrobione zostanie WOE (lepiej uciąglić dwie zmienne niż dyskretyzować prawie wszystkie)

In [17]:
# --- 1️⃣ Grupowanie PKD do sektorów ---
def kategoria_pkd(kod):
            if kod == 0:
                return "Nieznane / brak danych"
            elif 100 <= kod < 400:  # np. 1011, 142, 236
                return "Rolnictwo, leśnictwo, rybactwo"
            elif 500 <= kod < 1000:
                return "Górnictwo i wydobywanie"
            elif 1000 <= kod < 3500:  # np. 1623, 2562, 3320
                return "Przemysł i produkcja"
            elif 3500 <= kod < 4500:  # np. 3511, 4120, 4110
                return "Energetyka i budownictwo"
            elif 4500 <= kod < 4800:  # np. 4646, 4633, 4671
                return "Handel hurtowy i detaliczny"
            elif 4800 <= kod < 5500:
                return "Transport i magazynowanie"
            elif 5500 <= kod < 7000:
                return "Zakwaterowanie, gastronomia, IT, finanse"
            elif 7000 <= kod < 8000:
                return "Doradztwo, działalność profesjonalna"
            elif 8000 <= kod < 9000:
                return "Administracja, edukacja, zdrowie"
            else:
                return "Inne usługi"

df["pkdKod"] = df["pkdKod"].astype("object")
df["pkdKod"] = df["pkdKod"].apply(kategoria_pkd)

# --- 2️⃣ Grupowanie form własności ---
def kategoria_wlasnosci(kod):
            if kod in [111, 112, 113, 121, 122, 123, 131, 132, 133]:
                return "Sektor publiczny"
            elif kod in [214, 215, 224, 225]:
                return "Sektor prywatny krajowy"
            elif kod in [216, 226, 236]:
                return "Sektor prywatny zagraniczny"
            elif kod in [234, 235]:
                return "Sektor mieszany krajowy"
            elif kod == 0:
                return "Brak danych"
            else:
                return "Inna forma"

df["formaWlasnosci_Symbol"] = df["formaWlasnosci_Symbol"].astype("object")
df["formaWlasnosci_Symbol"] = df["formaWlasnosci_Symbol"].apply(
    kategoria_wlasnosci
)

Zastąpienie nieskończoności NaN-ami

In [18]:
df.replace(to_replace=[np.inf, -np.inf], value=np.nan, inplace=True)

Duży model językowy przygotował słownik kolumn, które mogą oraz które nie powinny być traktowane jako braki danych (True=[0<=>brak danych])

In [19]:
zero_as_missing = {
    'szczegolnaFormaPrawna_Symbol': False,
    'formaWlasnosci_Symbol': False,
    'pkdKod': False,
    'wsk_liczba_dni_istnienia': False,
    'Aktywa': True,
    'Aktywa_trwale': True,
    'Wartosci_niematerialne_prawne': True,
    'Wartosc_firmy': True,
    'Rzeczowe_aktywa_trwale': True,
    'Srodki_trwale': True,
    'Naleznosci_dlugoterminowe': True,
    'Inwestycje_dlugoterminowe': True,
    'Rozliczenia_miedzyokresowe_dlugie': True,
    'Aktywa_obrotowe': True,
    'Zapasy': False,
    'Naleznosci_krotkoterminowe': True,
    'Naleznosci_dostaw_uslug_12m_powiazane': True,
    'Naleznosci_dostaw_uslug_pow12m_powiazane': True,
    'Naleznosci_dostaw_uslug_12m_kapitale': True,
    'Naleznosci_dostaw_uslug_pow12m_kapitale': True,
    'Naleznosci_dostaw_uslug_12m_pozostale': True,
    'Naleznosci_dostaw_uslug_pow12m_pozostale': True,
    'Naleznosci_dostaw_uslug_pozostale_sadowe': True,
    'Inwestycje_krotkoterminowe': False,
    'Srodki_pieniezne': False,
    'Rozliczenia_miedzyokresowe_krotkie': False,
    'Kapital_wlasny': True,
    'Kapital_podstawowy': True,
    'Kapital_zapasowy': True,
    'Zysk_netto': False,
    'Zobowiazania_rezerwy': False,
    'Rezerwy_na_zobowiazania': False,
    'Rezerwa_z_tytulu_odroczonego_podatku_dochodowego': False,
    'Rezerwa_na_swiadczenia_emerytalne': False,
    'Rezerwa_na_swiadczenia_emerytalne_dlugie': False,
    'Rezerwa_na_swiadczenia_emerytalne_krotkie': False,
    'Pozostale_rezerwy': False,
    'Pozostale_rezerwy_krotkie': False,
    'Zobowiazania_dlugoterminowe': False,
    'Kredyty_pozyczki_dlugie': False,
    'Inne_zobowiazania_fin_dlugoterminowe': False,
    'Zobowiazania_krotkoterminowe': False,
    'Zobowiazania_dostaw_uslug_12m_powiazane': False,
    'Zobowiazania_dostaw_uslug_pow12m_powiazane': False,
    'Zobowiazania_dostaw_uslug_12m_kapitale': False,
    'Zobowiazania_dostaw_uslug_pow12m_kapitale': False,
    'Kredyty_pozyczki_krotkie': False,
    'Inne_zobowiazania_fin_krotkoterminowe': False,
    'Zobowiazania_dostaw_uslug_12m_pozostale': False,
    'Zobowiazania_dostaw_uslug_pow12m_pozostale': False,
    'Rozliczenia_miedzyokresowe_b': False,
    'Ujemna_wartosc_firmy': True,
    'Inne_rozliczenia_miedzyokresowe': False,
    'Inne_rozliczenia_miedzyokresowe_dlugie': False,
    'Inne_rozliczenia_miedzyokresowe_krotkie': False,
    'schemat_wsk_bilans': False,
    'Naleznosci_dostaw_uslug_12m': True,
    'Naleznosci_dostaw_uslug_pow12m': True,
    'Zobowiazania_dostaw_uslug_12m': True,
    'Zobowiazania_dostaw_uslug_pow12m': True,
    'Kredyty_pozyczki': False,
    'wsk_kapital_do_aktywa': True,
    'przychody_sprzedazy': True,
    'koszty_sprzedanych_produktow': False,
    'koszty_sprzedazy': False,
    'koszty_ogolnego_zarzadu': False,
    'zysk_sprzedazy': False,
    'pozostale_przychody_oper': False,
    'dotacje': False,
    'koszty_operacyjne_pozostale': False,
    'zysk_operacyjny': False,
    'przychody_finansowe': False,
    'dywidendy_udzialy': False,
    'przychody_odsetki': False,
    'koszty_finansowe': False,
    'koszty_odsetki': False,
    'zysk_brutto': False,
    'podatek_dochodowy': False,
    'zysk_netto': False,
    'koszty_operacyjne': False,
    'amortyzacja': True,
    'schemat_wsk_rzis': False,
    'przychody': True,
    'wsk_amortyzacja': True,
    'wsk_koszty_operacyjne': True,
    # Wskaźniki finansowe – traktujemy 0 jako brak danych
    'wsk_Zobowiazania_krotkoterminowe': True,
    'wsk_Zobowiazania_dlugoterminowe': True,
    'wsk_marza_brutto_1': True,
    'wsk_marza_brutto_2': True,
    'wsk_stopa_marzy_brutto': True,
    'wsk_rent_operacyjna': True,
    'wsk_ROS': True,
    'wsk_ROA': True,
    'wsk_s_ROA': True,
    'wsk_rent_operacyjna_aktywow': True,
    'wsk_ROE': True,
    'wsk_s_ROE': True,
    'wsk_mnoznik_kap_wl': True,
    'wsk_zwrot_aktywa_trwale': True,
    'wsk_rent_kaptial_podstawowy': True,
    'wsk_akt_generowania_got_1': True,
    'wsk_rent_sprzedazy': True,
    'wsk_ebit': True,
    'wsk_ebitda_1': True,
    'wsk_ebitda_2': True,
    'wsk_ebitda_3': True,
    'wsk_marza_ebitda_1': True,
    'wsk_marza_ebitda_2': True,
    'wsk_marza_ebitda_3': True,
    'wsk_marza_ebit': True,
    'wsk_ebitda_aktywa_1': True,
    'wsk_ebitda_aktywa_2': True,
    'wsk_ebitda_aktywa_3': True,
    'wsk_zwrot_aktywa_mat': True,
    'wsk_zysk_zobowiazania': True,
    'wsk_zysk_op_zobowiazania': True,
    'wsk_sprzedaz_kap_obrotowy': True,
    'wsk_koszty_przychody': True,
    'wsk_rent_kapitalu': True,
    'wsk_stopa_zysku_sprzedaz': True,
    'wsk_pokrycie_wyd_fin_gotowkowe_1': True,
    'wsk_koszt_długu_1': True,
    'wsk_koszt_długu_2': True,
    'wsk_pokrycie_aktywow_tr_kapitalem_st': True,
    'wsk_struktury_finansowania': True,
    'wsk_pokrycie_zob_kr_gotowkowe_1': True,
    'wsk_zysk_operacyjny_zob_1': True,
    'wsk_zysk_operacyjny_zob_2': True,
    'wsk_zadluzenia_gotowki_1': True,
    'wsk_koszty_fin_przychody': True,
    'wsk_koszty_odsetki_przychody': True,
    'wsk_zadluzenie_gotowka': True,
    'wsk_udzial_kap_wlasnego_aktywa_1': True,
    'wsk_udzial_kap_wlasnego_aktywa_2': True,
    'wsk_ogolnego_zadluzenia_1': True,
    'wsk_ogolnego_zadluzenia_2': True,
    'wsk_pokrycie_aktywow_kap_stalym': True,
    'wsk_zadluzenie_kap_wlasnego': True,
    'wsk_ogolnego_zadluzenia_pozyczki': True,
    'wsk_zadluzenia_pozyczki_dlugie': True,
    'wsk_zadluzenia_dlugie': True,
    'wsk_zadluzenia_krotkie': True,
    'wsk_pokrycia_zobowiazan_rz_aktywami_trwalymi': True,
    'wsk_ROE_brutto': True,
    'wsk_ROA_operacyjny': True,
    'wsk_efekt_dzwigni_fin_1': True,
    'wsk_efekt_dzwigni_fin_2': True,
    'wsk_pokrycia_odsetek_zyskiem': True,
    'wsk_ebitda_koszty_odsetkowe_1': True,
    'wsk_ebitda_koszty_odsetkowe_2': True,
    'wsk_ebitda_koszty_odsetkowe_3': True,
    'wsk_ebitda_koszty_finansowe_1': True,
    'wsk_ebitda_koszty_finansowe_2': True,
    'wsk_ebitda_koszty_finansowe_3': True,
    'wsk_ebitda_zobowiazan_odsetki_1': True,
    'wsk_ebitda_zobowiazan_odsetki_2': True,
    'wsk_ebitda_zobowiazan_odsetki_3': True,
    'wsk_ebitda_zobowiazan_odsetki_4': True,
    'wsk_ebitda_zobowiazan_1': True,
    'wsk_ebitda_zobowiazan_2': True,
    'wsk_ebitda_zobowiazan_3': True,
    'wsk_rotacja_aktywow_1': True,
    'wsk_rotacja_aktywow_2': True,
    'wsk_rotacja_rz_aktywow_trwalych': True,
    'wsk_rotacja_wartosci_niewaterialnych': True,
    'wsk_rotacja_zapasow': False,
    'wsk_rotacja_naleznosci': False,
    'wsk_rotacja_naleznosci_dostaw_uslug': False,
    'wsk_cykl_operacyjny': False,
    'wsk_poziom_kosztow_operacyjnych': False,
    'wsk_poziom_kosztow_finansowych': False,
    'wsk_obrotowsci_naleznosci': False,
    'wsk_rotacja_zobowiazan': False,
    'wsk_rotacja_zobowiazan_dostaw_uslug': False,
    'wsk_cykl_konwersji_gotowki': False,
    'wsk_plynnosc_biez_1': True,
    'wsk_plynnosc_biez_2': True,
    'wsk_plynnosc_biez_3': True,
    'wsk_plynnosc_szybka_1': False,
    'wsk_plynnosc_szybka_2': False,
    'wsk_plynnosc_gotowkowa_1': False,
    'wsk_poziom_kapitalu_obrotowego_netto': False,
    'wsk_udzial_kapitalu_obrotowego_netto': False,
    'wsk_udzial_zob_biez_sprzedaz_1': True,
    'wsk_udzial_zob_biez_sprzedaz_2': False,
    'wsk_udzial_zob_biez_aktywa_1': True,
    'wsk_udzial_zob_biez_aktywa_2': True,
    'wsk_udzial_zapasy_zobowiazania': False,
    'wsk_udzial_zapasy_kap_obrotowy': False,
    'wsk_udzial_kap_obrotowego_w_fin': False,
    'wsk_zysk_ebitda_1': True,
    'wsk_zysk_ebitda_2': True,
    'wsk_zysk_ebitda_3': True,
    'wsk_obrotowosc_gotowkowa': False,
    'wsk_struktura_majatku': True,
    'wsk_struktury_kapitalu': True,
    'wsk_zast_kapitalu_wlasnego': True,
    'wsk_zast_kapitalu_podstawowego': True,
    'wsk_zast_kapitalu_stalego': True,
    'wsk_zast_kapitalu_obcego': True,
    'wsk_sytuacji_fin': True,
    'wsk_struktura_kap_wlasnego_1': True,
    'wsk_struktura_kap_wlasnego_2': True,
    'wsk_struktura_kap_wlasnego_s_1': True,
    'wsk_struktura_kap_wlasnego_s_2': True,
    'wsk_zadluzenia': True,
    'wsk_zob_dlugoterminowe_aktywa_rzeczowe': True,
    'wsk_zob_oprocentowanych': True,
    'wsk_zob_oprocentowanych_aktywa_rzeczowe': True,
    'wsk_struktura_kap_obcego_s': True,
    'wsk_zob_s_aktywa_rzeczowe': True,
    'wsk_fin_majatku_kapitalem': True,
    'default': False
}


Lista kolumn, które nie powinny być zerami

In [20]:
shouldnt_be_0 = pd.Series(zero_as_missing.keys())[list(zero_as_missing.values())]

Zastąpienie w tych kolumnach zer NaN-ami (aby traktować te pozycje jako braki danych)

In [21]:
df[shouldnt_be_0] = df[shouldnt_be_0].replace(to_replace=0, value=np.nan)

Lista kolumn poszeregowana względem liczby braków danych

In [22]:
na_list = df.isna().sum().sort_values(ascending=False); na_list.head(15)

Naleznosci_dostaw_uslug_pow12m_kapitale     2999
Ujemna_wartosc_firmy                        2997
Naleznosci_dostaw_uslug_pow12m_powiazane    2995
Zobowiazania_dostaw_uslug_pow12m            2984
Wartosc_firmy                               2981
Naleznosci_dostaw_uslug_pow12m              2970
wsk_akt_generowania_got_2                   2962
wsk_zysk_CF_operacyjny                      2962
RP_przeplywy_operacyjne                     2962
wsk_zadluzenia_gotowki_2                    2962
wsk_pokrycie_zob_kr_gotowkowe_2             2962
wsk_pokrycie_wyd_fin_gotowkowe_2            2962
Naleznosci_dostaw_uslug_pow12m_pozostale    2959
Naleznosci_dostaw_uslug_12m_kapitale        2952
Naleznosci_dostaw_uslug_pozostale_sadowe    2943
dtype: int64

# Podsumowanie
Kolumny do wyrzucenia:
``` 
[
        "schemat_wsk_bilans",
        "schemat_wsk_rzis",
        "szczegolnaFormaPrawna_Symbol",
]
```

```
Kolumny do przerobienia przy pomocy słowników, a potem WOE: `["pkdKod", "formaWlasnosci_Symbol"]`
